In [ ]:
!pip install tensorflow

In [ ]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Update this path to where your files are stored in your Drive
DRIVE_PATH = '/content/drive/MyDrive/Music_AI'
MODEL_SAVE_PATH = '/content/drive/MyDrive/model_data'

os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

Mounted at /content/drive


parsing the midi files, use this path for your thing: 

In [ ]:
import os
import json
import warnings
import numpy as np
import tensorflow as tf
from music21 import converter, note, chord, interval, pitch
from google.colab import drive, runtime

# 1. SILENCE FILLED WARNINGS
# This stops the "TranslateWarning" from cluttering your console
warnings.filterwarnings("ignore", category=UserWarning, module='music21')
warnings.filterwarnings("ignore", message=".*Instrument.*")

# 2. Mount Google Drive
drive.mount('/content/drive')

# Constants
GRID = 0.25
SEQUENCE_LENGTH = 64
ACCEPTABLE_DURATIONS = [0.25, 0.5, 0.75, 1.0, 2.0, 3.0, 4.0]

def is_acceptable_note(event):
    return event.duration.quarterLength in ACCEPTABLE_DURATIONS

def transpose(score):
    try:
        key = score.analyze('key')
        target_tonic = pitch.Pitch('C') if key.mode == "major" else pitch.Pitch('A')
        itvl = interval.Interval(key.tonic, target_tonic)
        return score.transpose(itvl)
    except:
        return score # Return original if key analysis fails

def encode_song(song, time_step=GRID):
    encoded_song = []
    for event in song.flatten().notesAndRests:
        if not is_acceptable_note(event):
            continue

        if isinstance(event, note.Note):
            symbol = event.pitch.midi
        elif isinstance(event, note.Rest):
            symbol = "R"
        else:
            continue

        steps = int(event.duration.quarterLength / time_step)
        for step in range(steps):
            encoded_song.append(str(symbol) if step == 0 else "_")

    return " ".join(encoded_song)

def preprocess(midi_dir):
    songs = []
    midi_files = [f for f in os.listdir(midi_dir) if f.endswith(".mid") or f.endswith(".midi")]
    total_files = len(midi_files)

    print(f"--- Starting Preprocessing of {total_files} files ---")

    for index, file in enumerate(midi_files):
        # SHOW PROGRESS: Every song is printed with its index
        print(f"[{index + 1}/{total_files}] Processing: {file}")

        try:
            score = converter.parse(os.path.join(midi_dir, file))
            score = transpose(score)
            encoded = encode_song(score)
            if encoded.strip():
                songs.append(encoded)
        except Exception as e:
            print(f"   ! Error processing {file}: {e}")

    return songs

def create_single_file_dataset(songs, output_path, sequence_length):
    output_dir = os.path.dirname(output_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    delimiter = " / " * sequence_length
    dataset = delimiter.join(songs)

    with open(output_path, "w") as f:
        f.write(dataset.strip())
    return dataset

def create_mapping(dataset, mapping_path):
    vocab = list(set(dataset.split()))
    mapping = {symbol: i for i, symbol in enumerate(vocab)}
    with open(mapping_path, "w") as f:
        json.dump(mapping, f, indent=4)
    return mapping

def generate_training_sequences(dataset, mapping, sequence_length):
    print("Generating training sequences...")
    int_songs = [mapping[symbol] for symbol in dataset.split()]
    inputs, targets = [], []
    num_sequences = len(int_songs) - sequence_length

    for i in range(num_sequences):
        inputs.append(int_songs[i:i+sequence_length])
        targets.append(int_songs[i+sequence_length])

    vocab_size = len(mapping)

    # FORCE GPU USAGE
    print(f"Encoding {len(inputs)} sequences on GPU...")
    with tf.device('/GPU:0'):
        inputs = tf.keras.utils.to_categorical(inputs, num_classes=vocab_size)
        targets = np.array(targets)

    return inputs, targets

def main():
    # --- UPDATE THESE PATHS ---
    MIDI_DIR = "/content/drive/MyDrive/MusicAI/Midi_data/Midi_files"
    DATASET_PATH = "/content/drive/MyDrive/MusicAI/model_data/file_dataset.txt"
    MAPPING_PATH = "/content/drive/MyDrive/MusicAI/model_data/mapping.json"
    # --------------------------

    if not os.path.exists(MIDI_DIR):
        print(f"Error: MIDI directory '{MIDI_DIR}' not found. Check your Drive path.")
        return

    processed_songs = preprocess(MIDI_DIR)

    if not processed_songs:
        print("No songs were processed. Check your MIDI files.")
        return

    full_dataset = create_single_file_dataset(processed_songs, DATASET_PATH, SEQUENCE_LENGTH)
    symbol_mapping = create_mapping(full_dataset, MAPPING_PATH)

    X, y = generate_training_sequences(full_dataset, symbol_mapping, SEQUENCE_LENGTH)

    print("-" * 30)
    print(f"SUCCESS: Preprocessing Complete.")
    print(f"Final Data Shape - X: {X.shape}, y: {y.shape}")
    print("-" * 30)

    # DISCONNECT TO SAVE GPU LIMITS
    print("Disconnecting runtime to save your GPU quota...")
    runtime.unassign()

if __name__ == "__main__":
    main()

training the model

In [1]:

import os
import pickle
import warnings
import gc
import numpy as np
import tensorflow as tf
from google.colab import drive, runtime
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Embedding
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from music21 import converter, note, chord

# 1. INITIAL SETTINGS
warnings.filterwarnings("ignore", category=UserWarning, module='music21')
drive.mount('/content/drive')

# --- CONFIGURATION ---
MIDI_DIR = "/content/drive/MyDrive/MusicAI/Midi_data/Midi_files"
DATA_DIR = "/content/drive/MyDrive/MusicAI/model_data"
os.makedirs(DATA_DIR, exist_ok=True)

SEQUENCE_LENGTH = 100 # Slightly shorter sequences work better for smaller datasets
BATCH_SIZE = 128     # Fast training for 200 songs
GRID = 0.25

def quantize(duration):
    return round(duration / GRID) * GRID

def parse_midi_files(midi_folder, max_songs=200):
    # Separate cache for the 200-song version
    cache_path = os.path.join(DATA_DIR, f"raw_notes_{max_songs}.pkl")

    if os.path.exists(cache_path):
        print(f"--- Loading cached notes for {max_songs} songs ---")
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    notes = []
    files = [f for f in os.listdir(midi_folder) if f.lower().endswith((".mid", ".midi"))]
    files = files[:max_songs] # LIMIT TO 200 SONGS

    print(f"--- Parsing {len(files)} files ---")

    for i, filename in enumerate(files):
        if i % 20 == 0: print(f"Processing: {i}/{len(files)}...")
        try:
            score = converter.parse(os.path.join(midi_folder, filename), quantizePost=False)
            for element in score.flat.notesAndRests:
                d = quantize(element.duration.quarterLength)
                if d <= 0: continue
                if isinstance(element, note.Note):
                    notes.append(f"{element.pitch}_{d}")
                elif isinstance(element, chord.Chord):
                    pitches = ".".join(str(p) for p in sorted(element.pitches))
                    notes.append(f"{pitches}_{d}")
                elif isinstance(element, note.Rest) and d >= 0.5:
                    notes.append(f"rest_{d}")
        except: continue

    with open(cache_path, "wb") as f:
        pickle.dump(notes, f)
    return notes

def train():
    tf.keras.backend.clear_session()

    # Parse exactly 200 songs
    raw_notes = parse_midi_files(MIDI_DIR, max_songs=200)

    unique_notes = sorted(set(raw_notes))
    vocab_size = len(unique_notes)
    n2i = {n: i for i, n in enumerate(unique_notes)}

    with open(os.path.join(DATA_DIR, "note_to_int.pkl"), "wb") as f:
        pickle.dump(n2i, f)

    encoded = np.array([n2i[n] for n in raw_notes], dtype=np.int32)
    total_sequences = len(encoded) - SEQUENCE_LENGTH
    steps_per_epoch = total_sequences // BATCH_SIZE

    del raw_notes
    gc.collect()

    # DATASET PIPELINE
    dataset = tf.data.Dataset.from_tensor_slices(encoded)
    dataset = dataset.window(SEQUENCE_LENGTH + 1, shift=1, drop_remainder=True)
    dataset = dataset.flat_map(lambda w: w.batch(SEQUENCE_LENGTH + 1))
    dataset = dataset.map(lambda chunk: (chunk[:-1], chunk[-1]), num_parallel_calls=tf.data.AUTOTUNE)

    # Larger shuffle buffer is fine for 200 songs
    dataset = dataset.shuffle(5000).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

    # MODEL ARCHITECTURE (Optimized for 200 songs)
    model = Sequential([
        Embedding(vocab_size, 512), # Larger embedding for better feature capture
        LSTM(512, return_sequences=True),
        Dropout(0.4), # Increased dropout to prevent overfitting on 200 songs
        LSTM(512),
        Dropout(0.4),
        Dense(256, activation='relu'),
        Dense(vocab_size, activation='softmax')
    ])

    model.compile(loss="sparse_categorical_crossentropy", optimizer=tf.keras.optimizers.Adam(0.0005))
    checkpoint_path = os.path.join(DATA_DIR, "music_model_200.h5")

    # ADDED: EarlyStopping to stop if the model stops improving
    early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

    print(f"\nTraining on {total_sequences} sequences (200 Songs).")

    try:
        model.fit(
            dataset,
            epochs=80, # More epochs because 200 songs train faster
            steps_per_epoch=steps_per_epoch,
            callbacks=[
                ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='loss'),
                ReduceLROnPlateau(monitor='loss', factor=0.2, patience=5),
                early_stop
            ]
        )
    except Exception as e:
        print(f"Error: {e}")
    finally:
        print("Done. Disconnecting...")
        runtime.unassign()

if __name__ == "__main__":
    train()

Mounted at /content/drive
--- Parsing 199 files ---
Processing: 0/199...
Processing: 20/199...
Processing: 40/199...
Processing: 60/199...
Processing: 80/199...
Processing: 100/199...
Processing: 120/199...
Processing: 140/199...
Processing: 160/199...
Processing: 180/199...

Training on 666713 sequences (200 Songs).
Epoch 1/80
5208/5208 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 5.6560

5208/5208 ━━━━━━━━━━━━━━━━━━━━ 107s 19ms/step - loss: 5.6560 - learning_rate: 5.0000e-04
Epoch 2/80


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


5208/5208 ━━━━━━━━━━━━━━━━━━━━ 1s 116us/step - loss: 0.0000e+00 - learning_rate: 5.0000e-04
Epoch 3/80
5208/5208 ━━━━━━━━━━━━━━━━━━━━ 99s 19ms/step - loss: 5.0306 - learning_rate: 5.0000e-04
Epoch 4/80
5208/5208 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step - loss: 0.0000e+00 - learning_rate: 5.0000e-04
Epoch 5/80
5208/5208 ━━━━━━━━━━━━━━━━━━━━ 98s 19ms/step - loss: 4.7630 - learning_rate: 5.0000e-04
Epoch 6/80
5208/5208 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step - loss: 0.0000e+00 - learning_rate: 5.0000e-04
Epoch 7/80
5208/5208 ━━━━━━━━━━━━━━━━━━━━ 102s 19ms/step - loss: 4.5486 - learning_rate: 5.0000e-04
Epoch 8/80
5208/5208 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step - loss: 0.0000e+00 - learning_rate: 1.0000e-04
Epoch 9/80
5208/5208 ━━━━━━━━━━━━━━━━━━━━ 100s 19ms/step - loss: 4.5570 - learning_rate: 1.0000e-04
Epoch 10/80
5208/5208 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step - loss: 0.0000e+00 - learning_rate: 1.0000e-04
Epoch 11/80
5208/5208 ━━━━━━━━━━━━━━━━━━━━ 99s 19ms/step - loss: 4.4020 - learning_rate: 1.0000e-04
Epoch